In [1]:
# ============================================================
# EDA COMPLETA — RELATÓRIO ESTRUTURADO PARA IA
# ------------------------------------------------------------
# Objetivo: extrair TUDO que um agente precisa saber sobre a
# base antes de decidir o pré-processamento e a modelagem.
# ============================================================
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 250)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# Recalcula BASE_LONG se não estiver definido
if 'BASE_LONG' not in dir():
    _cwd = Path.cwd()
    _base = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    BASE_LONG = _base / 'dados' / 'processado' / 'base_longitudinal_rr_2019_2025.parquet'

df = pd.read_parquet(BASE_LONG)

ALVOS      = ['ABANDONO_FUN_T1', 'ABANDONO_FUN_AF_T1', 'ABANDONO_MED_T1']
META       = ['ANO', 'ANO_ALVO']
IDS        = ['CO_ENTIDADE', 'NO_ENTIDADE', 'CO_MUNICIPIO', 'NO_MUNICIPIO']
CATEGORICAS_ESPERADAS = ['CO_MUNICIPIO', 'LOCALIZACAO', 'DEPENDENCIA']

def secao(n, titulo):
    print('\n' + '=' * 78)
    print(f'  {n}. {titulo}')
    print('=' * 78)

def sub(t):
    print(f'\n--- {t} ---')

# ============================================================
# 1. FORMA E TIPOS
# ============================================================
secao(1, 'FORMA E TIPOS')
print(f'Linhas  : {len(df):,}')
print(f'Colunas : {df.shape[1]}')
print('\nDistribuição de dtypes:')
print(df.dtypes.value_counts().to_string())

num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include=['string', 'object']).columns.tolist()

sub(f'Colunas numéricas ({len(num_cols)})')
print(num_cols)
sub(f'Colunas categóricas/texto ({len(cat_cols)})')
print(cat_cols)

# ============================================================
# 2. COBERTURA TEMPORAL
# ============================================================
secao(2, 'COBERTURA TEMPORAL')
print('Anos presentes em ANO      :', sorted(df['ANO'].dropna().unique().tolist()))
print('Anos presentes em ANO_ALVO :', sorted(df['ANO_ALVO'].dropna().unique().tolist()))

sub('Pares (ANO → ANO_ALVO) e tamanho')
pares = df.groupby(['ANO', 'ANO_ALVO'], dropna=False).size().rename('n').reset_index()
print(pares.to_string(index=False))

sub('Escolas únicas por ANO_ALVO')
print(df.groupby('ANO_ALVO')['CO_ENTIDADE'].nunique().to_string())

# ============================================================
# 3. VALORES AUSENTES
# ============================================================
secao(3, 'VALORES AUSENTES')
nulos = pd.DataFrame({
    'n_nulos' : df.isna().sum(),
    'pct_nulos': (df.isna().mean() * 100).round(1),
}).sort_values('pct_nulos', ascending=False)
print(nulos.to_string())

# ============================================================
# 4. COLUNAS CONSTANTES OU QUASE CONSTANTES
# ============================================================
secao(4, 'COLUNAS CONSTANTES OU QUASE CONSTANTES')
constantes = []
quase = []
for c in df.columns:
    n_unique = df[c].nunique(dropna=True)
    if n_unique <= 1:
        constantes.append(c)
    elif n_unique <= 3 and df[c].dtype.kind in 'ifb':
        quase.append((c, n_unique, df[c].value_counts(dropna=False).head(3).to_dict()))
print(f'Totalmente constantes: {constantes}')
print(f'\nQuase constantes (≤3 valores):')
for c, n, vals in quase:
    print(f'  {c}: {n} únicos → {vals}')

# ============================================================
# 5. CARDINALIDADE DE CATEGÓRICAS
# ============================================================
secao(5, 'CARDINALIDADE DAS CATEGÓRICAS')
for c in cat_cols:
    n_unique = df[c].nunique(dropna=True)
    print(f'\n{c}: {n_unique} únicos')
    vc = df[c].value_counts(dropna=False).head(10)
    print(vc.to_string())

# ============================================================
# 6. ESTATÍSTICAS DESCRITIVAS DAS NUMÉRICAS
# ============================================================
secao(6, 'ESTATÍSTICAS DAS NUMÉRICAS')
desc = df[num_cols].describe().T
desc['nulos'] = df[num_cols].isna().sum().values
desc['n_unicos'] = df[num_cols].nunique().values
print(desc.round(4).to_string())

# ============================================================
# 7. ANÁLISE DOS ALVOS
# ============================================================
secao(7, 'ANÁLISE DOS ALVOS')
for alvo in ALVOS:
    sub(f'Alvo: {alvo}')
    print(f'Não nulos: {df[alvo].notna().sum():,} ({df[alvo].notna().mean()*100:.1f}%)')
    print('Por ANO_ALVO:')
    tab = df.groupby('ANO_ALVO')[alvo].agg(
        n='count', media='mean', mediana='median', std='std',
        p75=lambda s: s.quantile(0.75), max='max'
    ).round(3)
    print(tab.to_string())
    print(f'Zeros no alvo: {(df[alvo] == 0).sum():,} ({(df[alvo] == 0).mean()*100:.1f}%)')

# ============================================================
# 8. ALVO EM ESCOLAS PÚBLICAS vs PRIVADAS
# ============================================================
secao(8, 'ALVO POR REDE (PÚBLICA vs PRIVADA)')
if 'REDE_PUBLICA' in df.columns:
    for alvo in ALVOS:
        sub(f'{alvo} — por REDE_PUBLICA')
        tab = df.groupby('REDE_PUBLICA')[alvo].agg(
            n='count', media='mean', mediana='median', max='max'
        ).round(3)
        print(tab.to_string())

# ============================================================
# 9. ALVO POR LOCALIZAÇÃO E DEPENDÊNCIA
# ============================================================
secao(9, 'ALVO POR LOCALIZACAO / DEPENDENCIA')
for cat in ['LOCALIZACAO', 'DEPENDENCIA']:
    if cat in df.columns:
        sub(f'{cat} × alvos (média)')
        tab = df.groupby(cat)[ALVOS].agg(['count', 'mean']).round(3)
        print(tab.to_string())

# ============================================================
# 10. CORRELAÇÃO DAS NUMÉRICAS COM OS ALVOS
# ============================================================
secao(10, 'CORRELAÇÃO COM OS ALVOS')
# Pearson — só numéricas, ignora nulos par a par
corr = {}
for alvo in ALVOS:
    corr[alvo] = df[num_cols].corrwith(df[alvo].astype('float64'), method='pearson')
corr_df = pd.DataFrame(corr).round(4)
# Ordena pelo alvo principal (FUN_AF)
if 'ABANDONO_FUN_AF_T1' in corr_df.columns:
    corr_df = corr_df.reindex(corr_df['ABANDONO_FUN_AF_T1'].abs().sort_values(ascending=False).index)
print(corr_df.to_string())

# ============================================================
# 11. DETECÇÃO DE POSSÍVEL LEAKAGE
# ============================================================
secao(11, 'POSSÍVEL LEAKAGE')
# Alvos "irmãos": mesma variável derivada, mesmo período
irmaos = [a for a in ALVOS]
sub('Alvos irmãos (mesmo período, variável derivada)')
for a in irmaos:
    print(f'  {a} — se o alvo for outro, remover')

sub('Colunas derivadas do próprio alvo (mesmo ano, mesmo segmento)')
sugestoes_leak = []
for c in num_cols:
    # mesmo segmento do alvo, sem sufixo de tempo
    for alvo in ALVOS:
        if c != alvo and c.replace('_T1', '') in alvo.replace('_T1', ''):
            sugestoes_leak.append((c, alvo, 'mesmo segmento, mesmo período'))
print('Detecção automática:', sugestoes_leak if sugestoes_leak else 'nenhuma óbvia')

sub('Colunas com correlação > 0.95 com algum alvo')
for alvo in ALVOS:
    altas = corr_df[alvo].dropna().abs()
    altas = altas[altas > 0.95].index.tolist()
    print(f'  {alvo}: {altas}')

# ============================================================
# 12. DUPLICATAS
# ============================================================
secao(12, 'DUPLICATAS')
dup_full = df.duplicated().sum()
print(f'Linhas completamente duplicadas: {dup_full}')
dup_key = df.duplicated(subset=['CO_ENTIDADE', 'ANO']).sum()
print(f'Duplicatas por (CO_ENTIDADE, ANO): {dup_key}')
if dup_key > 0:
    dups = df[df.duplicated(subset=['CO_ENTIDADE', 'ANO'], keep=False)]
    print(dups[['CO_ENTIDADE', 'NO_ENTIDADE', 'ANO', 'ANO_ALVO']].head(20).to_string(index=False))

# ============================================================
# 13. OUTLIERS (IQR) POR COLUNA NUMÉRICA
# ============================================================
secao(13, 'OUTLIERS (IQR × 3)')
outliers = []
for c in num_cols:
    s = df[c].dropna().astype('float64')
    if len(s) == 0:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    n_out = ((s < lo) | (s > hi)).sum()
    if n_out > 0:
        outliers.append((c, n_out, round(100 * n_out / len(s), 2), round(lo, 2), round(hi, 2)))
outliers.sort(key=lambda t: -t[1])
print(f'{"coluna":<28} {"n_out":>7} {"%":>7} {"lo":>10} {"hi":>10}')
for c, n, p, lo, hi in outliers[:40]:
    print(f'{c:<28} {n:>7} {p:>6.2f}% {lo:>10} {hi:>10}')

# ============================================================
# 14. COBERTURA DE VARIÁVEIS CRÍTICAS POR ANO_ALVO
# ============================================================
secao(14, 'COBERTURA DE VARIÁVEIS CRÍTICAS POR ANO_ALVO')
criticas = [
    'ATU_FUN_AF', 'TDI_FUN_AF', 'IED_FUN_ALTO',
    'APROVACAO_FUN_AF', 'REPROVACAO_FUN_AF', 'ABANDONO_FUN_AF',
    'ATU_MED_TOTAL', 'TDI_MED_TOTAL', 'IED_MED_ALTO',
    'APROVACAO_MED', 'REPROVACAO_MED', 'ABANDONO_MED',
]
criticas = [c for c in criticas if c in df.columns]
cob = df.groupby('ANO_ALVO')[criticas].apply(lambda g: g.notna().mean().round(3))
print(cob.to_string())

# ============================================================
# 15. DISTRIBUIÇÃO TEMPORAL DO ALVO PRINCIPAL
# ============================================================
secao(15, 'EVOLUÇÃO TEMPORAL DO ALVO PRINCIPAL (ABANDONO_FUN_AF_T1)')
alvo_p = 'ABANDONO_FUN_AF_T1'
if alvo_p in df.columns:
    tab = df.groupby('ANO_ALVO')[alvo_p].agg(
        n='count', media='mean', mediana='median',
        p10=lambda s: s.quantile(0.10),
        p90=lambda s: s.quantile(0.90),
        max='max'
    ).round(3)
    print(tab.to_string())

# ============================================================
# 16. ESCOLAS QUE APARECEM EM TODOS OS ANOS
# ============================================================
secao(16, 'PAINEL BALANCEADO?')
n_por_escola = df.groupby('CO_ENTIDADE')['ANO'].nunique()
print('Escolas por nº de anos distintos que aparecem:')
print(n_por_escola.value_counts().sort_index().to_string())
print(f'\nTotal de escolas únicas: {df["CO_ENTIDADE"].nunique():,}')
print(f'Escolas em todos os 7 anos: {(n_por_escola == 7).sum():,}')

# ============================================================
# 17. RESUMO E RECOMENDAÇÕES
# ============================================================
secao(17, 'RESUMO ESTRUTURADO (para IA)')
print(f'''
CONTEXTO
- Unidade         : escola-ano
- Período         : ANO {df["ANO"].min()}–{df["ANO"].max()} → alvo em ANO_ALVO {df["ANO_ALVO"].min()}–{df["ANO_ALVO"].max()}
- Escolas únicas  : {df["CO_ENTIDADE"].nunique()}
- Linhas          : {len(df)}

ALVOS DISPONÍVEIS
- ABANDONO_FUN_T1     : n={df["ABANDONO_FUN_T1"].notna().sum()}, média={df["ABANDONO_FUN_T1"].mean():.3f}
- ABANDONO_FUN_AF_T1  : n={df["ABANDONO_FUN_AF_T1"].notna().sum()}, média={df["ABANDONO_FUN_AF_T1"].mean():.3f}   ← alvo principal
- ABANDONO_MED_T1     : n={df["ABANDONO_MED_T1"].notna().sum()}, média={df["ABANDONO_MED_T1"].mean():.3f}

PROBLEMAS IDENTIFICADOS
1. ANO_ALVO ausente em {df["ANO_ALVO"].isna().sum()} linhas ({df["ANO_ALVO"].isna().mean()*100:.1f}%) → último ano da série, sem alvo.
2. Missing estrutural: variáveis de um segmento só existem em escolas que oferecem aquele segmento.
3. Colunas totalmente constantes: {constantes}
4. Duplicatas por (CO_ENTIDADE, ANO): {dup_key}

RECOMENDAÇÕES PARA O PRÉ-PROCESSAMENTO
- Filtrar REDE_PUBLICA == 1 antes de modelar (o modelo é sobre rede pública).
- Separar em dois modelos: anos finais (alvo ABANDONO_FUN_AF_T1) e médio (alvo ABANDONO_MED_T1).
- Remover de features os alvos irmãos (não usar ABANDONO_FUN_T1 nem ABANDONO_MED_T1 quando o alvo for ABANDONO_FUN_AF_T1, e vice-versa).
- Imputação: mediana para numéricas, 'Missing' para categóricas.
- Encoding: OneHot para LOCALIZACAO, DEPENDENCIA, CO_MUNICIPIO.
- Validação temporal: treino ANO ≤ 2022, validação 2023, teste 2024→2025.
- Escalar numéricas: SIM (Ridge precisa; árvores não se importam).
''')

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.